In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torchvision.transforms as transforms

class BHMSDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.folder = folder
        self.transform = transform
        self.samples = []
        self.classes = sorted(set(f.split("-")[0] for f in os.listdir(folder) if f.endswith(".png")))
        self.classes_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for fname in os.listdir(folder):
            if fname.endswith(".png"):
                cls = fname.split("-")[0]
                self.samples.append((os.path.join(folder, fname), self.classes_to_idx[cls]))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        path, label = self.samples[index]
        img = Image.open(path).convert("L")
        if self.transform:
            img = self.transform(img)
        return img, label

In [2]:
class MathSymbolCNN(nn.Module):
    def __init__(self, num_classes):
        super(MathSymbolCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),     # 28 x 28 -> 14 x 14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)      # 14 x 14 -> 7 x 7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [3]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = BHMSDataset("data/symbols", transform=transform)

print(f"Classes: {dataset.classes}")
print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")

Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'dot', 'minus', 'plus', 'slash', 'w', 'x', 'y', 'z']
Total images: 27000
Number of classes: 18


In [4]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MathSymbolCNN(num_classes=len(dataset.classes)).to(device=device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = (correct / total) * 100

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = (val_correct / val_total) * 100

    print(f"Epoch {epoch + 1} of {EPOCHS} Loss: {train_loss / len(train_loader)} Train accuracy: {train_accuracy} Val accuracy: {val_accuracy}")

    torch.save({
        "model_state": model.state_dict(),
        "classes": dataset.classes,
        "num_classes": len(dataset.classes),
    }, "math_symbol_cnn_v4.pth")

Device: cuda
Epoch 1 of 15 Loss: 0.9276879606162303 Train accuracy: 71.06481481481481 Val accuracy: 91.05555555555556
Epoch 2 of 15 Loss: 0.3716302369914111 Train accuracy: 88.58333333333334 Val accuracy: 93.98148148148148
Epoch 3 of 15 Loss: 0.285901198152607 Train accuracy: 90.69444444444444 Val accuracy: 94.77777777777779
Epoch 4 of 15 Loss: 0.2400270301033054 Train accuracy: 92.30555555555556 Val accuracy: 94.96296296296296
Epoch 5 of 15 Loss: 0.21424744366365073 Train accuracy: 93.3611111111111 Val accuracy: 95.98148148148148
Epoch 6 of 15 Loss: 0.19474159247798503 Train accuracy: 93.94907407407408 Val accuracy: 96.27777777777777
Epoch 7 of 15 Loss: 0.18021814393419663 Train accuracy: 94.06481481481481 Val accuracy: 95.57407407407408
Epoch 8 of 15 Loss: 0.17225604181414877 Train accuracy: 94.5 Val accuracy: 96.27777777777777
Epoch 9 of 15 Loss: 0.15104360239216563 Train accuracy: 95.21296296296296 Val accuracy: 96.9074074074074
Epoch 10 of 15 Loss: 0.14642307799331536 Train accura

In [5]:
print(f"class to index: {dataset.classes_to_idx}")

class to index: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'dot': 10, 'minus': 11, 'plus': 12, 'slash': 13, 'w': 14, 'x': 15, 'y': 16, 'z': 17}
